# Corrected Polymarket Trading Features

## Purpose

Prepare the trading activity features for the final competitive asset prediction models. The financial predictors, target variable and 428 competitive events remain unchanged.

Trading activity is classified as support or opposition using both YES and NO tokens. All features are calculated on the daily calendar and lagged before merging with the financial data.

## Input files

- `Final_Competitive_Markets_MODEL_READY_Dataset.xlsx`
- `Polymarket_Selected_Markets_Historical_Trades.csv.gz`
- `Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features.xlsx` (reference dataset)

Run with the historical transaction file to calculate daily activity from trades. If only the final workbook is available, the saved daily activity sheet can be used to recalculate the lagged predictors and check the final model inputs.

*Repository note: this implementation was recreated for the portfolio using the dissertation methodology and archived data. The original correction notebook is unavailable.*

## Step 1: Import Libraries and Set File Paths

Place the historical input files in the notebook folder, or update `DATA` to point to the data directory.

In [ ]:
# If needed in Colab, uncomment: %pip install pandas numpy openpyxl
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Put the files in the current notebook directory (or /content in Colab).
DATA = Path('.')
MODE = 'auto'  # 'raw', 'checkpoint' or 'auto' (prefer raw if available)
# Change these paths if your filenames or folders differ.
BASE = DATA / 'Final_Competitive_Markets_MODEL_READY_Dataset.xlsx'
RAW = DATA / 'Polymarket_Selected_Markets_Historical_Trades.csv.gz'
REFERENCE = DATA / 'Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features.xlsx'
OUTPUT = DATA / 'Replicated_CORRECTED_PM_Trading_Features.xlsx'

def find_file(path, pattern):
    if path.is_file():
        return path
    matches = sorted(DATA.glob(pattern))
    if len(matches) > 1:
        raise ValueError(f'Multiple matches for {pattern}: {matches}. Set the exact path above.')
    return matches[0] if matches else path

BASE = find_file(BASE, 'Final_Competitive_Markets_MODEL_READY_Dataset*.xlsx')
RAW = find_file(RAW, 'Polymarket_Selected_Markets_Historical_Trades*.csv*')
REFERENCE = find_file(REFERENCE, 'Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features*.xlsx')
assert MODE in {'raw', 'checkpoint', 'auto'}
mode = ('raw' if RAW.is_file() else 'checkpoint') if MODE == 'auto' else MODE
if mode == 'raw' and not RAW.is_file():
    raise FileNotFoundError(RAW)
if mode == 'checkpoint' and not REFERENCE.is_file():
    raise FileNotFoundError('Checkpoint mode requires the archived corrected workbook.')
assert OUTPUT.resolve() not in {BASE.resolve(), REFERENCE.resolve()}, 'Never overwrite source files.'
print('Selected route:', mode)

## Step 2: Load the Model-Ready Dataset

Keep the financial returns, probability features and winner definition from the preceding notebook. If the base workbook is unavailable, use the same base columns from the saved final dataset.

Create a daily calendar for each alternative, including weekends and the 15-day warm-up period.

In [ ]:
BASE_COLUMNS = [
    'Market_ID', 'Event_ID', 'Date', 'Next_Financial_Date', 'Asset',
    'PM_Prob_Current_Audit', 'Price_t', 'Price_Next', 'Forward_Log_Return', 'Winner',
    'Lag1_Return', 'Lag2_Return', 'Lag3_Return', 'Financial_Prev1_Date',
    'Financial_Prev2_Date', 'Financial_Prev3_Date', 'PM_Signal_Date_Lag1',
    'PM_Prob_Lag1', 'PM_Change_1D_Lag1', 'PM_Change_From_Start_Lag1',
    'Days_to_Resolution', 'Days_to_Resolution_Years', 'PM_Prob_x_TimeToResolution'
]
KEY = ['Market_ID', 'Asset', 'Date']
base_path = BASE if BASE.is_file() else REFERENCE
if not base_path.is_file():
    raise FileNotFoundError('Supply the original base dataset or archived corrected workbook.')
master = pd.read_excel(base_path, sheet_name='Model_Ready_Master')[BASE_COLUMNS].copy()
for col in [c for c in BASE_COLUMNS if 'Date' in c]:
    master[col] = pd.to_datetime(master[col])
master = master.sort_values(['Market_ID', 'Date', 'Event_ID', 'Asset']).reset_index(drop=True)
assert len(master) == 1012 and master.Event_ID.nunique() == 428
assert not master.duplicated(['Event_ID', 'Asset']).any()
assert master.groupby('Event_ID').Winner.sum().eq(1).all()
assert (master.PM_Signal_Date_Lag1 < master.Date).all()
print('Base source:', base_path.name, '| rows:', len(master), '| events:', master.Event_ID.nunique())

# Preserve the original 15-day warm-up and daily calendar, including weekends.
calendar = pd.concat([
    pd.DataFrame({'Market_ID': market, 'Asset': asset,
                  'Date': pd.date_range(g.Date.min() - pd.Timedelta(days=15), g.Date.max())})
    for market, g in master.groupby('Market_ID') for asset in sorted(g.Asset.unique())
], ignore_index=True)
assert len(calendar) == 1585

## Step 3: Classify and Aggregate Trading Activity

| Trade | Direction for the YES candidate |
|---|---|
| BUY YES | Support |
| SELL YES | Opposition |
| BUY NO | Opposition |
| SELL NO | Support |

In the binary markets, opposition to Bitcoin is support for Gold. In the three-way market, opposition to an asset is not assigned to another asset.

Calculate notional as trade size multiplied by executed price. Express NO prices as their YES-equivalent prices when calculating price dispersion.

In [ ]:
CONTRACTS = {
    '0x6830f5868d794655bc325a8e414fae1944755f513796a498aba43d8911b22e91': ('BTC_GOLD_2025', 'Bitcoin'),
    '0xdcae9573d5680a2c958cca4676ed28df7a06861572124f3396cc60ab6b92b9c2': ('BTC_GOLD_2026', 'Bitcoin'),
    '0xb276435811dc77171602f790db2b5900e780adfadb7cff57e547d58fb1a8215f': ('BTC_GOLD_SP500', 'Bitcoin'),
    '0x23db4bef50396fe700d23cb2eaa7d726da9280f7ce3ecd54126a976e9591c7bf': ('BTC_GOLD_SP500', 'Gold'),
    '0x5290e1debd1d164f7828f32c9da7747018462293928a8624e27a070b7255f1f1': ('BTC_GOLD_SP500', 'SP500'),
}
EXPECTED_COUNTS = dict(zip(CONTRACTS, [2226, 2857, 4020, 2820, 2131]))

def clean_trades(raw):
    # Accept either the original cleaned CSV export or equivalent API field names.
    t = raw.rename(columns={old: new for old, new in {
        'conditionId':'Condition_ID', 'asset':'Token_ID', 'side':'Side',
        'size':'Trade_Size_Shares', 'price':'Trade_Price', 'timestamp':'Timestamp'
    }.items() if new not in raw.columns}).copy()
    needed = ['Condition_ID','Token_ID','Side','Trade_Size_Shares','Trade_Price',
              'Timestamp','outcome','transactionHash','proxyWallet']
    if not set(needed).issubset(t.columns):
        raise ValueError(f'Missing trade columns: {set(needed)-set(t.columns)}')
    t['Condition_ID'] = t.Condition_ID.astype(str).str.lower()
    t['Token_ID'] = t.Token_ID.astype(str)
    t['Side'] = t.Side.str.upper()
    t['outcome'] = t.outcome.str.upper()
    for col in ['Trade_Size_Shares','Trade_Price','Timestamp']:
        t[col] = pd.to_numeric(t[col], errors='raise')
    assert t[needed[:-1]].notna().all().all()
    assert t.Side.isin(['BUY','SELL']).all() and t.outcome.isin(['YES','NO']).all()
    assert t.Trade_Size_Shares.ge(0).all() and t.Trade_Price.between(0,1).all()
    assert t.Condition_ID.isin(CONTRACTS).all(), 'Unexpected contract ID'
    t = t.drop_duplicates(needed, keep='last').copy()
    t['Date'] = pd.to_datetime(t.Timestamp, unit='s', utc=True).dt.tz_localize(None).dt.normalize()
    t['Derived_Notional'] = t.Trade_Size_Shares * t.Trade_Price
    t['Market_ID'] = t.Condition_ID.map(lambda c: CONTRACTS[c][0])
    t['Candidate'] = t.Condition_ID.map(lambda c: CONTRACTS[c][1])
    t['Supports_Yes'] = ((t.Side == 'BUY') & (t.outcome == 'YES')) | ((t.Side == 'SELL') & (t.outcome == 'NO'))
    t['Candidate_Price'] = np.where(t.outcome == 'YES', t.Trade_Price, 1-t.Trade_Price)
    return t

DAILY_COLUMNS = ['Support_Notional','Support_Share_Volume','Support_Trade_Count',
    'Support_Unique_Takers','Support_Mean_Trade_Notional','Support_Median_Trade_Notional',
    'Support_HHI','Support_Top_Share','Support_Effective_Takers','Contract_Notional',
    'Contract_Trade_Count','Contract_Unique_Takers','Support_Notional_Contract',
    'Opposition_Notional_Contract','Net_Support_Imbalance','Net_Support_Flow',
    'Candidate_Price_Dispersion','Candidate_Price_Range']
ZERO_COLS = ['Support_Notional','Support_Share_Volume','Support_Trade_Count','Support_Unique_Takers',
    'Contract_Notional','Contract_Trade_Count','Contract_Unique_Takers','Support_Notional_Contract',
    'Opposition_Notional_Contract','Net_Support_Imbalance','Net_Support_Flow']

def aggregate_candidate(g, supports):
    s = g.loc[supports]
    support = s.Derived_Notional.sum()
    total = g.Derived_Notional.sum()
    opposing = g.loc[~supports, 'Derived_Notional'].sum()
    by_wallet = s.groupby('proxyWallet').Derived_Notional.sum()
    shares = by_wallet / by_wallet.sum() if by_wallet.sum() > 0 else pd.Series(dtype=float)
    hhi = (shares**2).sum() if len(shares) else np.nan
    p, w = g.Candidate_Price, g.Derived_Notional
    dispersion = np.sqrt(np.average((p - np.average(p, weights=w))**2, weights=w)) if total > 0 else np.nan
    return dict(Support_Notional=support, Support_Share_Volume=s.Trade_Size_Shares.sum(),
        Support_Trade_Count=len(s), Support_Unique_Takers=s.proxyWallet.nunique(),
        Support_Mean_Trade_Notional=s.Derived_Notional.mean(), Support_Median_Trade_Notional=s.Derived_Notional.median(),
        Support_HHI=hhi, Support_Top_Share=shares.max(), Support_Effective_Takers=1/hhi if hhi > 0 else np.nan,
        Contract_Notional=total, Contract_Trade_Count=len(g), Contract_Unique_Takers=g.proxyWallet.nunique(),
        Support_Notional_Contract=support, Opposition_Notional_Contract=opposing,
        Net_Support_Imbalance=(support-opposing)/total if total > 0 else 0., Net_Support_Flow=support-opposing,
        Candidate_Price_Dispersion=dispersion, Candidate_Price_Range=p.max()-p.min())

def daily_from_trades(t, calendar):
    rows = []
    for (market, candidate, date), g in t.groupby(['Market_ID','Candidate','Date']):
        if market == 'BTC_GOLD_SP500':
            directions = [(candidate, g.Supports_Yes)]
        else:
            directions = [('Bitcoin', g.Supports_Yes), ('Gold', ~g.Supports_Yes)]
        for asset, supports in directions:
            rows.append(dict(Market_ID=market, Asset=asset, Date=date, **aggregate_candidate(g,supports)))
    daily = calendar.merge(pd.DataFrame(rows), on=KEY, how='left', validate='one_to_one')
    daily[ZERO_COLS] = daily[ZERO_COLS].fillna(0)
    return daily[KEY + DAILY_COLUMNS]

if mode == 'raw':
    t = clean_trades(pd.read_csv(RAW, dtype={'Token_ID':str,'asset':str}))
    assert len(t) == 14054, f'Expected 14,054 archived trades, got {len(t)}'
    assert t.Condition_ID.value_counts().to_dict() == EXPECTED_COUNTS, 'Contract counts differ from archived download'
    limits = master.groupby('Market_ID').Date.agg(['min','max'])
    for market, g in t.groupby('Market_ID'):
        assert g.Date.between(limits.loc[market,'min']-pd.Timedelta(days=15), limits.loc[market,'max']).all()
    daily0 = daily_from_trades(t, calendar)
    raw_count = len(t)
else:
    daily0 = pd.read_excel(REFERENCE, sheet_name='Daily_PM_Refined_Activity')[KEY+DAILY_COLUMNS]
    daily0['Date'] = pd.to_datetime(daily0.Date)
    raw_count = 14054  # archived extraction count, not independently recalculated in this route
    print('Using archived daily aggregates (no raw trade verification in checkpoint mode).')

assert not daily0.duplicated(KEY).any()
assert len(daily0) == len(calendar)
assert calendar.merge(daily0[KEY], on=KEY, how='left', indicator=True)._merge.eq('both').all()
print('Daily calendar rows:', len(daily0))

## Step 4: Create Lagged Trading Features

Use the previous calendar day for activity features. Calculate seven-day volume, volume surge, cumulative support and relative support shares separately for each alternative.

Volume surge is the difference between the previous day's log support notional and its mean over the preceding seven days. If all alternatives have zero volume, assign equal relative shares. Leave undefined concentration and mean trade size as missing.

In [ ]:
def lag_features(daily0):
    d = daily0.sort_values(KEY).reset_index(drop=True).copy()
    group = ['Market_ID','Asset']
    sources = {
        'Date':'PM_Refined_Activity_Signal_Date_Lag1',
        'Support_Notional':'PM_Support_Notional_Lag1',
        'Support_Trade_Count':'PM_Support_Trade_Count_Lag1',
        'Support_Unique_Takers':'PM_Support_Unique_Takers_Lag1',
        'Support_Mean_Trade_Notional':'PM_Support_Mean_Trade_Notional_Lag1',
        'Support_HHI':'PM_Support_HHI_Lag1',
        'Support_Top_Share':'PM_Support_Top_Taker_Share_Lag1',
        'Contract_Notional':'PM_Contract_Notional_Lag1',
        'Contract_Trade_Count':'PM_Contract_Trade_Count_Lag1',
        'Contract_Unique_Takers':'PM_Contract_Unique_Takers_Lag1',
        'Net_Support_Imbalance':'PM_Net_Support_Imbalance_Lag1',
        'Net_Support_Flow':'PM_Net_Support_Flow_Lag1',
        'Candidate_Price_Dispersion':'PM_Candidate_Price_Dispersion_Lag1',
        'Candidate_Price_Range':'PM_Candidate_Price_Range_Lag1'
    }
    for src, dst in sources.items():
        d[dst] = d.groupby(group)[src].shift(1)
    for feature in ['Support_Notional','Support_Trade_Count','Support_Unique_Takers','Support_Mean_Trade_Notional','Contract_Notional']:
        d[f'PM_Log_{feature}_Lag1'] = np.log1p(d[f'PM_{feature}_Lag1'])
    flow = d.PM_Net_Support_Flow_Lag1
    d['PM_SignedLog_Net_Support_Flow_Lag1'] = np.sign(flow)*np.log1p(abs(flow))
    lag = d.groupby(group)['PM_Support_Notional_Lag1']
    d['PM_Support_Notional_7D_Lag1'] = lag.transform(lambda v: v.rolling(7,min_periods=1).sum())
    d['PM_Log_Support_Notional_7D_Lag1'] = np.log1p(d.PM_Support_Notional_7D_Lag1)
    d['PM_Support_Volume_Surge_Lag1'] = d.PM_Log_Support_Notional_Lag1 - d.groupby(group).PM_Log_Support_Notional_Lag1.transform(lambda v: v.shift(1).rolling(7,min_periods=1).mean())
    d['PM_Cumulative_Support_Notional_Lag1'] = d.groupby(group).Support_Notional.transform(lambda v: v.cumsum().shift(1))
    d['PM_Log_Cumulative_Support_Notional_Lag1'] = np.log1p(d.PM_Cumulative_Support_Notional_Lag1)
    for col, dest in [('PM_Support_Notional_Lag1','PM_Relative_Support_Volume_Share_Lag1'),
                      ('PM_Cumulative_Support_Notional_Lag1','PM_Relative_Cumulative_Support_Share_Lag1'),
                      ('PM_Contract_Notional_Lag1','PM_Relative_Contract_Volume_Share_Lag1')]:
        total = d.groupby(['Market_ID','Date'])[col].transform('sum')
        n = d.groupby(['Market_ID','Date'])[col].transform('size')
        d[dest] = np.where(total > 0, d[col]/total, 1/n)
    d['PM_Contract_Active_Lag1'] = d.PM_Contract_Notional_Lag1.gt(0).astype(int)
    d['PM_Support_Active_Lag1'] = d.PM_Support_Notional_Lag1.gt(0).astype(int)
    return d

daily = lag_features(daily0)
assert len(daily) == 1585 and daily.shape[1] == 51
valid = daily.PM_Refined_Activity_Signal_Date_Lag1.notna()
assert (daily.loc[valid,'PM_Refined_Activity_Signal_Date_Lag1'] < daily.loc[valid,'Date']).all()

## Step 5: Merge the Trading Features

Join on market, asset and forecast date. Retain the original observations and check that every event has exactly one winner. Create the probability and contract-volume interaction.

In [ ]:
ACTIVITY_COLUMNS = [
    'PM_Refined_Activity_Signal_Date_Lag1','PM_Support_Notional_Lag1',
    'PM_Log_Support_Notional_Lag1','PM_Relative_Support_Volume_Share_Lag1',
    'PM_Log_Cumulative_Support_Notional_Lag1','PM_Relative_Cumulative_Support_Share_Lag1',
    'PM_Log_Support_Unique_Takers_Lag1','PM_Net_Support_Imbalance_Lag1',
    'PM_SignedLog_Net_Support_Flow_Lag1','PM_Support_Volume_Surge_Lag1',
    'PM_Log_Contract_Notional_Lag1','PM_Prob_x_Log_Contract_Notional_Lag1',
    'PM_Relative_Contract_Volume_Share_Lag1','PM_Support_HHI_Lag1',
    'PM_Log_Support_Mean_Trade_Notional_Lag1','PM_Candidate_Price_Dispersion_Lag1',
    'PM_Contract_Active_Lag1','PM_Support_Active_Lag1'
]
interaction = 'PM_Prob_x_Log_Contract_Notional_Lag1'
corrected = master.merge(daily[KEY + [c for c in ACTIVITY_COLUMNS if c != interaction]],
                         on=KEY, how='left', validate='one_to_one')
corrected[interaction] = corrected.PM_Prob_Lag1 * corrected.PM_Log_Contract_Notional_Lag1
corrected = corrected[BASE_COLUMNS + ACTIVITY_COLUMNS]
assert corrected.shape == (1012,41)
assert corrected.Event_ID.nunique() == 428
assert corrected.groupby('Event_ID').Winner.sum().eq(1).all()
assert not corrected.duplicated(['Event_ID','Asset']).any()
assert (corrected.PM_Refined_Activity_Signal_Date_Lag1 < corrected.Date).all()
required = [c for c in ACTIVITY_COLUMNS if c not in ['PM_Support_HHI_Lag1','PM_Log_Support_Mean_Trade_Notional_Lag1','PM_Candidate_Price_Dispersion_Lag1']]
assert corrected[required].notna().all().all()
print(corrected.groupby('Market_ID').Event_ID.nunique())

## Step 6: Validate Against the Final Dataset

Check the daily activity table and model-ready dataset against the saved dissertation workbook. Compare dates and labels exactly, and numeric values within floating-point tolerance.

Using the saved daily aggregates verifies the feature calculations and merge, not the original trade-level aggregation.

In [ ]:
def compare_frames(actual, expected, key):
    a = actual.sort_values(key).reset_index(drop=True)
    b = expected.sort_values(key).reset_index(drop=True)
    assert len(a) == len(b) and set(a.columns) == set(b.columns)
    results = []
    for col in b:
        if pd.api.types.is_numeric_dtype(b[col]):
            ok = np.isclose(a[col].astype(float), b[col].astype(float), rtol=1e-10, atol=1e-10, equal_nan=True)
        else:
            ok = (a[col].eq(b[col]) | (a[col].isna() & b[col].isna())).to_numpy()
        results.append({'Column':col, 'Matching_Rows':int(ok.sum()), 'Rows':len(ok), 'Match':bool(ok.all())})
    return pd.DataFrame(results)

if REFERENCE.is_file():
    daily_check = compare_frames(daily, pd.read_excel(REFERENCE,sheet_name='Daily_PM_Refined_Activity'), KEY)
    master_check = compare_frames(corrected, pd.read_excel(REFERENCE,sheet_name='Model_Ready_Master'), KEY)
    for label, check in [('Daily', daily_check), ('Master', master_check)]:
        print(label, f'{check.Match.sum()}/{len(check)} columns match')
        display(check.loc[~check.Match])
    assert daily_check.Match.all() and master_check.Match.all(), 'Archived data comparison failed; do not replace the original.'
else:
    print('No archive supplied: structural checks only; numerical replication not verified.')

## Step 7: Export the Dataset

Save the processed data and quality checks separately from the archived workbook. Use the original final workbook for the modelling notebooks unless the regenerated data have been checked against it.

In [ ]:
# Summarise zero and missing values by market.
zero_features = [c for c in ACTIVITY_COLUMNS if c not in [
    'PM_Refined_Activity_Signal_Date_Lag1','PM_SignedLog_Net_Support_Flow_Lag1',
    'PM_Prob_x_Log_Contract_Notional_Lag1','PM_Contract_Active_Lag1','PM_Support_Active_Lag1']]
zero_rows = []
for label, g in [('ALL',corrected)] + list(corrected.groupby('Market_ID')):
    for col in zero_features:
        zero_rows.append(dict(Market_ID=label, Feature=col, Rows=len(g), Zero_Rows=int(g[col].eq(0).sum()),
                             Missing_Rows=int(g[col].isna().sum()), Zero_Pct=g[col].eq(0).mean()))
zero_analysis = pd.DataFrame(zero_rows)
qc = pd.DataFrame({'Check':['Rows','Competitive_Event_IDs','BTC_GOLD_2025_Events','BTC_GOLD_2026_Events','BTC_GOLD_SP500_Events',
                           'Raw_Trades','All_Winners_Valid','No_Duplicate_Event_Asset','Refined_Activity_Timing_Valid'],
                   'Value':[len(corrected),corrected.Event_ID.nunique(),
                            *[corrected.loc[corrected.Market_ID==m,'Event_ID'].nunique() for m in ['BTC_GOLD_2025','BTC_GOLD_2026','BTC_GOLD_SP500']],
                            raw_count,1,1,1]})
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    corrected.to_excel(writer,sheet_name='Model_Ready_Master',index=False)
    daily.to_excel(writer,sheet_name='Daily_PM_Refined_Activity',index=False)
    if REFERENCE.is_file():
        pd.read_excel(REFERENCE,sheet_name='Feature_Guide').to_excel(writer,sheet_name='Feature_Guide',index=False)
    qc.to_excel(writer,sheet_name='QC_Summary',index=False)
    zero_analysis.to_excel(writer,sheet_name='Zero_Analysis',index=False)
    pd.DataFrame({'Route':[mode], 'Raw_count_independently_checked':[mode=='raw'],
                  'Archived_comparison_performed':[REFERENCE.is_file()], 'Source':['Portfolio implementation of dissertation methodology']}).to_excel(writer,sheet_name='Replication_Audit',index=False)
print('Saved:', OUTPUT)